# 🧠 Pipeline: Backpropagation desde Cero

**Propósito:** Implementar una red neuronal *sin Keras* usando solo NumPy para entender
a fondo el algoritmo de backpropagation, la regla de la cadena y el descenso por gradiente.

---
## 📐 Regla de la Cadena (Chain Rule)

La retropropagación aplica la regla de la cadena del cálculo diferencial:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial w}$$

Donde:
- $L$ = pérdida (loss)
- $\hat{y}$ = predicción
- $z$ = combinación lineal (pre-activación)
- $w$ = peso

---
## 🔄 Forward Pass vs Backward Pass

| Fase | Qué hace | Dirección |
|------|----------|-----------|
| **Forward** | Calcula predicciones $\hat{y} = \sigma(XW + b)$ | Entrada → Salida |
| **Backward** | Calcula gradientes $\frac{\partial L}{\partial w}$ usando regla de la cadena | Salida → Entrada |
| **Update** | Ajusta pesos $w = w - \eta \cdot \nabla w$ | — |

In [ ]:
# MACHOTE — Setup universal del pipeline
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))
from modulo4_libreria import *

INFO = setup_completo()

In [ ]:
# MACHOTE — Dataset sintético no linealmente separable (make_moons)
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=300, noise=0.15, random_state=42)

print(f"Dataset: {X.shape[0]} muestras, {X.shape[1]} features")
print(f"Clases: {np.unique(y)}")

plt.figure(figsize=(6,5))
plt.scatter(X[y==0,0], X[y==0,1], c='steelblue', label='Clase 0', alpha=0.7, edgecolors='k')
plt.scatter(X[y==1,0], X[y==1,1], c='tomato', label='Clase 1', alpha=0.7, edgecolors='k')
plt.title('Dataset Make Moons — No linealmente separable')
plt.xlabel('Feature 1'); plt.ylabel('Feature 2')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

---
## 🧱 Implementación Manual (NumPy)

Construimos una red con 1 capa oculta (4 neuronas, activación sigmoide) y 1 neurona de salida (sigmoide).
Todas las funciones se implementan a mano para visualizar cada paso del backpropagation.

In [ ]:
# MACHOTE — Funciones de activación y pérdida

def sigmoid(x):
    """Sigmoide con estabilidad numérica."""
    x = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x))


def sigmoid_derivada(a):
    """Derivada de la sigmoide: σ'(x) = σ(x) * (1 - σ(x))"""
    return a * (1.0 - a)


def binary_cross_entropy(y_true, y_pred):
    """Binary Cross-Entropy Loss con clipping para estabilidad."""
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1.0 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

In [ ]:
# MACHOTE — Inicialización de parámetros

def initialize_params(n_input, n_hidden, n_output):
    """Inicializa pesos y biases con valores aleatorios pequeños."""
    np.random.seed(42)
    params = {
        'W1': np.random.randn(n_input, n_hidden) * 0.5,
        'b1': np.zeros((1, n_hidden)),
        'W2': np.random.randn(n_hidden, n_output) * 0.5,
        'b2': np.zeros((1, n_output))
    }
    print("Parámetros inicializados:")
    for k, v in params.items():
        print(f"   {k}: {v.shape}")
    return params


params = initialize_params(2, 4, 1)

In [ ]:
# MACHOTE — Forward Pass

def forward(X, params):
    """Forward propagation: calcula predicción capa por capa.
    
    Returns:
        dict: cache con Z1, A1, Z2, A2 (predicción final)
    """
    W1, b1 = params['W1'], params['b1']
    W2, b2 = params['W2'], params['b2']
    
    Z1 = X @ W1 + b1        # (n, 4)
    A1 = sigmoid(Z1)         # (n, 4)
    Z2 = A1 @ W2 + b2        # (n, 1)
    A2 = sigmoid(Z2)         # (n, 1) — predicción final
    
    return {'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2}


cache = forward(X, params)
print(f"Predicción inicial (primeras 5): {cache['A2'][:5].flatten().round(3)}")
print(f"Loss inicial: {binary_cross_entropy(y.reshape(-1,1), cache['A2']):.4f}")

In [ ]:
# MACHOTE — Backward Pass (retropropagación manual)

def backward(X, y, params, cache):
    """Backpropagation: calcula gradientes usando la regla de la cadena.
    
    Derivadas:
      dL/dZ2 = A2 - y                        (BCE + sigmoide combinados)
      dL/dW2 = A1.T @ dL/dZ2
      dL/db2 = sum(dL/dZ2, axis=0)
      dL/dA1 = dL/dZ2 @ W2.T
      dL/dZ1 = dL/dA1 * sigmoide'(A1)
      dL/dW1 = X.T @ dL/dZ1
      dL/db1 = sum(dL/dZ1, axis=0)
    """
    m = X.shape[0]
    W2 = params['W2']
    
    A1, A2 = cache['A1'], cache['A2']
    
    dZ2 = A2 - y.reshape(-1, 1)              # gradiente de BCE+sigmoide
    dW2 = (A1.T @ dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m
    
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * sigmoid_derivada(A1)
    dW1 = (X.T @ dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m
    
    return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}


grads = backward(X, y, params, cache)
print("Gradientes calculados:")
for k, v in grads.items():
    print(f"   {k}: forma={v.shape}, media={v.mean():.6f}")

In [ ]:
# MACHOTE — Actualización de parámetros (Gradient Descent)

def update_params(params, grads, lr=0.5):
    """Actualiza todos los parámetros: w = w - lr * dw"""
    params['W1'] -= lr * grads['dW1']
    params['b1'] -= lr * grads['db1']
    params['W2'] -= lr * grads['dW2']
    params['b2'] -= lr * grads['db2']
    return params


params = update_params(params, grads, lr=0.5)
print("Parámetros actualizados (1 paso de gradiente descendente) ✓")

In [ ]:
# MACHOTE — Bucle de entrenamiento completo

def entrenar(X, y, n_hidden=4, epochs=5000, lr=0.5, verbose=True):
    """Entrena una red neuronal manual con backpropagation."""
    n_input, n_output = X.shape[1], 1
    params = initialize_params(n_input, n_hidden, n_output)
    
    historial_loss = []
    y = y.reshape(-1, 1)
    
    for epoch in range(epochs):
        cache = forward(X, params)
        loss = binary_cross_entropy(y, cache['A2'])
        historial_loss.append(loss)
        
        grads = backward(X, y, params, cache)
        params = update_params(params, grads, lr)
        
        if verbose and (epoch + 1) % 1000 == 0:
            print(f"Época {epoch+1:4d} | Loss: {loss:.4f}")
    
    return params, historial_loss


print("Entrenando red manual...")
params_entrenados, hist_loss = entrenar(X, y, epochs=5000, lr=0.5)
print(f"\n✅ Entrenamiento completado. Loss final: {hist_loss[-1]:.4f}")

In [ ]:
# MACHOTE — Visualizar pérdida durante entrenamiento

plt.figure(figsize=(8,4))
plt.plot(hist_loss, color='steelblue', lw=1.5)
plt.title('Evolución de la pérdida (Backpropagation manual)')
plt.xlabel('Época'); plt.ylabel('Binary Cross-Entropy')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# MACHOTE — Frontera de decisión antes vs después

def plot_decision_boundary(modelo, X, y, title=''):
    """Dibuja la frontera de decisión de un clasificador binario."""
    x_min, x_max = X[:,0].min() - 0.5, X[:,0].max() + 0.5
    y_min, y_max = X[:,1].min() - 0.5, X[:,1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                          np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    
    if callable(modelo):
        Z = modelo(grid)
    else:
        cache = forward(grid, modelo)
        Z = cache['A2']
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, levels=[0, 0.5, 1], alpha=0.3, colors=['tomato', 'steelblue'])
    plt.scatter(X[y==0,0], X[y==0,1], c='steelblue', label='Clase 0', alpha=0.7, edgecolors='k')
    plt.scatter(X[y==1,0], X[y==1,1], c='tomato', label='Clase 1', alpha=0.7, edgecolors='k')
    plt.title(title)
    plt.xlabel('Feature 1'); plt.ylabel('Feature 2')
    plt.legend(); plt.grid(alpha=0.3)


fig, axes = plt.subplots(1, 2, figsize=(14,5))

plt.sca(axes[0])
params_ini = initialize_params(2, 4, 1)
plot_decision_boundary(params_ini, X, y, 'ANTES de entrenar')

plt.sca(axes[1])
plot_decision_boundary(params_entrenados, X, y, 'DESPUÉS de entrenar')

plt.tight_layout()
plt.show()

In [ ]:
# MACHOTE — Evaluación: precisión del modelo manual

cache = forward(X, params_entrenados)
y_pred = (cache['A2'] > 0.5).astype(int).flatten()
acc = np.mean(y_pred == y)

print(f"🎯 Accuracy del modelo manual: {acc:.4f} ({acc*100:.2f}%)")
print(f"\n¿Separa correctamente las lunas? {'✅ Sí' if acc > 0.85 else '⚠️ Parcialmente'}")

---
## ⚡ Comparación con Keras MLP

La misma arquitectura en Keras requiere ~5 líneas:

```python
modelo = keras.Sequential([
    layers.Dense(4, activation='sigmoid', input_shape=(2,)),
    layers.Dense(1, activation='sigmoid')
])
modelo.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
modelo.fit(X, y, epochs=100, verbose=0)
```

| Aspecto | Manual (NumPy) | Keras |
|---------|---------------|-------|
| Código | ~100 líneas | ~5 líneas |
| Optimizador | SGD fijo | Adam (adaptativo) |
| Velocidad | Lenta (CPU) | Optimizada (GPU) |
| Control | Total (cada gradiente) | Abstracto |
| Debug | Fácil (todo visible) | Caja negra |

> **Conclusión:** Implementar backpropagation manual es clave para entender *cómo* funciona el aprendizaje profundo.
> En producción siempre se usa Keras/PyTorch.

---
## ✅ Resumen de conceptos

1. **Forward pass:** $z = XW + b$, $a = \sigma(z)$ — propagamos hacia adelante
2. **Loss:** $\mathcal{L} = -\frac{1}{m}\sum [y\log(\hat{y}) + (1-y)\log(1-\hat{y})]$
3. **Backward pass:** Aplicamos regla de la cadena desde la salida hasta la entrada
4. **Gradient descent:** $w = w - \eta \cdot \frac{\partial \mathcal{L}}{\partial w}$
5. **Repetir:** Forward → Loss → Backward → Update

---
*Pipeline creado para el Diplomado en Redes Neuronales — Módulo 4* 🧠

In [ ]:
# TODO: Experimenta con diferentes arquitecturas
# - Cambia n_hidden a 8, 16 neuronas
# - Prueba con activación 'tanh' en lugar de sigmoide
# - Agrega una segunda capa oculta
# Pregunta: ¿Cómo cambia la frontera de decisión?